# Project Ares — Daily Trading Report
Run all cells in order. Cell 8 is your daily report.

**First time:** Run cells 1–7 to set up.

**Daily use:** Run Cell 1 (install), then jump to Cell 8 (report).

In [ ]:
# Cell 1: Install packages
!pip install -q pandas numpy yfinance pandas-ta

# Create folder structure
import os
os.makedirs('Ares/config', exist_ok=True)
os.makedirs('Ares/engine', exist_ok=True)
os.makedirs('Ares/data/ohlcv', exist_ok=True)
os.makedirs('Ares/logs', exist_ok=True)

# Create __init__.py
with open('Ares/engine/__init__.py', 'w') as f:
    pass

# Change working directory
os.chdir('Ares')
print('Setup complete. Working directory:', os.getcwd())

In [ ]:
# Cell 2: Config files
import json

# watchlist.json
watchlist = {
    "description": "Stocks to scan daily",
    "symbols": [
        "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
        "META", "TSLA", "JPM", "V", "JNJ",
        "SPY", "QQQ", "IWM"
    ]
}
with open('config/watchlist.json', 'w') as f:
    json.dump(watchlist, f, indent=4)

# strategy_params.json
params = {
    "rsi_oversold": 40,
    "rsi_overbought": 70,
    "min_vol_ratio": 1.5,
    "macd_threshold": 0,
    "lookback_days": 252
}
with open('config/strategy_params.json', 'w') as f:
    json.dump(params, f, indent=4)

# risk_rules.json
rules = {
    "_comment": "YOUR RULES. Follow these when trading.",
    "max_position_pct": 0.10,
    "max_concurrent_positions": 8,
    "cash_buffer_pct": 0.30,
    "stop_loss_multiplier": 2.0,
    "max_weekly_loss_pct": 0.05,
    "execution_mode": "paper"
}
with open('config/risk_rules.json', 'w') as f:
    json.dump(rules, f, indent=4)

print('Config files created.')

In [ ]:
%%writefile engine/data_feed.py
import yfinance as yf
import pandas as pd
from pathlib import Path

DATA_DIR = Path(__file__).parent.parent / "data" / "ohlcv"
DATA_DIR.mkdir(parents=True, exist_ok=True)

def download_stock(symbol, period="2y"):
    """Download OHLCV data for a stock and cache it locally."""
    df = yf.download(symbol, period=period)
    filepath = DATA_DIR / f"{symbol}.csv"
    df.to_csv(filepath)
    return df

def load_stock(symbol):
    """Load cached stock data from disk."""
    filepath = DATA_DIR / f"{symbol}.csv"
    if not filepath.exists():
        return download_stock(symbol)
    df = pd.read_csv(filepath, index_col=0, parse_dates=True, date_format='ISO8601')
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def refresh_watchlist(symbols):
    """Download fresh data for all stocks in watchlist."""
    data = {}
    for symbol in symbols:
        data[symbol] = download_stock(symbol)
    return data

In [ ]:
%%writefile engine/indicators.py
import pandas as pd
import pandas_ta as ta

def add_indicators(df):
    """Add all technical indicators to a DataFrame."""
    df['rsi'] = ta.rsi(df['Close'], length=14)

    df['vol_avg_20'] = df['Volume'].rolling(window=20).mean()
    df['vol_ratio'] = df['Volume'] / df['vol_avg_20']

    macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    df['macd'] = macd.iloc[:, 0]
    df['macd_signal'] = macd.iloc[:, 1]
    df['macd_hist'] = macd.iloc[:, 2]

    df['stdev_20'] = df['Close'].pct_change().rolling(20).std()

    df['high_52w'] = df['High'].rolling(252).max()
    df['pct_from_high'] = (df['Close'] - df['high_52w']) / df['high_52w']

    return df

In [ ]:
%%writefile engine/signals.py
import json
from pathlib import Path
from engine.data_feed import load_stock
from engine.indicators import add_indicators

def load_strategy_params():
    config_path = Path(__file__).parent.parent / "config" / "strategy_params.json"
    with open(config_path) as f:
        return json.load(f)

def check_rsi_reversal(symbol, df, params):
    """Check if stock shows RSI mean-reversion signal."""
    latest = df.iloc[-1]
    prev = df.iloc[-2]

    if latest['rsi'] >= params['rsi_oversold']:
        return None
    if latest['vol_ratio'] < params['min_vol_ratio']:
        return None
    if prev['rsi'] < params['rsi_oversold']:
        return None

    return {
        'symbol': symbol,
        'strategy': 'rsi_reversal',
        'date': str(latest.name)[:10],
        'price': round(float(latest['Close']), 2),
        'rsi': round(float(latest['rsi']), 1),
        'vol_ratio': round(float(latest['vol_ratio']), 2),
        'stdev_20': round(float(latest['stdev_20']), 4),
        'strength': 'high' if latest['rsi'] < 25 else 'medium'
    }

def check_momentum_breakout(symbol, df, params):
    """Check if stock is breaking to new 52-week high on volume."""
    latest = df.iloc[-1]

    if latest['pct_from_high'] < -0.01:
        return None
    if latest['vol_ratio'] < params['min_vol_ratio']:
        return None
    if latest['macd'] < latest['macd_signal']:
        return None

    return {
        'symbol': symbol,
        'strategy': 'momentum_breakout',
        'date': str(latest.name)[:10],
        'price': round(float(latest['Close']), 2),
        'rsi': round(float(latest['rsi']), 1),
        'vol_ratio': round(float(latest['vol_ratio']), 2),
        'stdev_20': round(float(latest['stdev_20']), 4),
        'strength': 'high' if latest['vol_ratio'] > 3.0 else 'medium'
    }

def scan_universe(symbols=None):
    """Scan all stocks in watchlist for signals."""
    params = load_strategy_params()

    if symbols is None:
        watchlist_path = Path(__file__).parent.parent / "config" / "watchlist.json"
        with open(watchlist_path) as f:
            symbols = json.load(f)['symbols']

    signals = []
    for symbol in symbols:
        try:
            df = load_stock(symbol)
            df = add_indicators(df)

            signal = check_rsi_reversal(symbol, df, params)
            if signal:
                signals.append(signal)
                continue

            signal = check_momentum_breakout(symbol, df, params)
            if signal:
                signals.append(signal)
        except Exception as e:
            print(f"Error scanning {symbol}: {e}")

    return signals

In [ ]:
%%writefile engine/tracker.py
import json
import csv
from datetime import datetime
from pathlib import Path
from engine.data_feed import load_stock
from engine.indicators import add_indicators

LOGS_DIR = Path(__file__).parent.parent / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)
TRADES_FILE = LOGS_DIR / "virtual_trades.json"

def load_trades():
    if not TRADES_FILE.exists():
        return []
    with open(TRADES_FILE) as f:
        return json.load(f)

def save_trades(trades):
    with open(TRADES_FILE, 'w') as f:
        json.dump(trades, f, indent=2)

def open_trade(signal):
    trades = load_trades()
    for t in trades:
        if t['symbol'] == signal['symbol'] and t['status'] == 'open':
            return

    portfolio = 10000
    position_size = portfolio * 0.10
    shares = position_size / signal['price']
    stop_loss = signal['price'] - (signal['price'] * signal['stdev_20'] * 2)

    trade = {
        'symbol': signal['symbol'],
        'strategy': signal['strategy'],
        'entry_date': signal['date'],
        'entry_price': signal['price'],
        'shares': round(shares, 2),
        'position_size': round(position_size, 2),
        'stop_loss': round(stop_loss, 2),
        'rsi_at_entry': signal['rsi'],
        'vol_at_entry': signal['vol_ratio'],
        'strength': signal['strength'],
        'status': 'open',
        'exit_date': None,
        'exit_price': None,
        'exit_reason': None,
        'pnl': None,
        'pnl_pct': None
    }

    trades.append(trade)
    save_trades(trades)

def check_open_trades():
    trades = load_trades()
    updated = False

    for trade in trades:
        if trade['status'] != 'open':
            continue

        try:
            df = load_stock(trade['symbol'])
            df = add_indicators(df)
            latest = df.iloc[-1]

            current_price = float(latest['Close'])
            current_rsi = float(latest['rsi'])
            today = str(latest.name)[:10]

            if current_price <= trade['stop_loss']:
                trade['status'] = 'closed'
                trade['exit_date'] = today
                trade['exit_price'] = trade['stop_loss']
                trade['exit_reason'] = 'stop_loss'
                pnl = (trade['stop_loss'] - trade['entry_price']) * trade['shares']
                trade['pnl'] = round(pnl, 2)
                trade['pnl_pct'] = round(
                    (trade['stop_loss'] - trade['entry_price'])
                    / trade['entry_price'] * 100, 2)
                updated = True

            elif current_rsi > 50 and today != trade['entry_date']:
                trade['status'] = 'closed'
                trade['exit_date'] = today
                trade['exit_price'] = round(current_price, 2)
                trade['exit_reason'] = 'target_reached'
                pnl = (current_price - trade['entry_price']) * trade['shares']
                trade['pnl'] = round(pnl, 2)
                trade['pnl_pct'] = round(
                    (current_price - trade['entry_price'])
                    / trade['entry_price'] * 100, 2)
                updated = True

        except Exception as e:
            print(f"  Error checking {trade['symbol']}: {e}")

    if updated:
        save_trades(trades)

    return trades

def print_scorecard():
    trades = load_trades()
    closed = [t for t in trades if t['status'] == 'closed']
    open_trades = [t for t in trades if t['status'] == 'open']

    print(f"\n[5] SCORECARD")
    print("-" * 40)

    if not closed and not open_trades:
        print("  No trades recorded yet.")
        return

    if open_trades:
        print(f"\n  OPEN POSITIONS ({len(open_trades)}):")
        for t in open_trades:
            try:
                df = load_stock(t['symbol'])
                current = float(df.iloc[-1]['Close'])
                unrealized = (current - t['entry_price']) / t['entry_price'] * 100
                arrow = "+" if unrealized > 0 else "-"
                print(f"    {t['symbol']}: entry ${t['entry_price']} -> "
                      f"now ${current:.2f} {arrow}{abs(unrealized):.1f}% | "
                      f"stop: ${t['stop_loss']}")
            except Exception:
                print(f"    {t['symbol']}: entry ${t['entry_price']} | "
                      f"stop: ${t['stop_loss']}")

    if closed:
        wins = [t for t in closed if t['pnl'] > 0]
        losses = [t for t in closed if t['pnl'] <= 0]
        total_pnl = sum(t['pnl'] for t in closed)
        avg_win = (sum(t['pnl_pct'] for t in wins) / len(wins)
                   if wins else 0)
        avg_loss = (sum(t['pnl_pct'] for t in losses) / len(losses)
                    if losses else 0)

        print(f"\n  CLOSED TRADES ({len(closed)}):")
        print(f"    Win rate:   {len(wins)}/{len(closed)} "
              f"({len(wins)/len(closed)*100:.0f}%)")
        print(f"    Total P&L:  ${total_pnl:+.2f}")
        print(f"    Avg win:    {avg_win:+.1f}%")
        print(f"    Avg loss:   {avg_loss:+.1f}%")

        print(f"\n  RECENT TRADES:")
        for t in closed[-5:]:
            icon = "W" if t['pnl'] > 0 else "L"
            print(f"    [{icon}] {t['symbol']} | "
                  f"{t['entry_date']} -> {t['exit_date']} | "
                  f"{t['pnl_pct']:+.1f}% | {t['exit_reason']}")

        strategies = {}
        for t in closed:
            s = t['strategy']
            if s not in strategies:
                strategies[s] = {'wins': 0, 'total': 0}
            strategies[s]['total'] += 1
            if t['pnl'] > 0:
                strategies[s]['wins'] += 1

        print(f"\n  BY STRATEGY:")
        for s, data in strategies.items():
            wr = data['wins'] / data['total'] * 100
            print(f"    {s}: {data['wins']}/{data['total']} wins ({wr:.0f}%)")

def export_csv():
    trades = load_trades()
    if not trades:
        print("  No trades to export.")
        return

    csv_path = LOGS_DIR / "trades_report.csv"
    columns = [
        'symbol', 'strategy', 'entry_date', 'entry_price', 'shares',
        'position_size', 'stop_loss', 'rsi_at_entry', 'vol_at_entry',
        'strength', 'status', 'exit_date', 'exit_price', 'exit_reason',
        'pnl', 'pnl_pct'
    ]

    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for t in trades:
            row = {col: t.get(col, '') for col in columns}
            writer.writerow(row)

    print(f"  CSV exported: logs/trades_report.csv")

In [ ]:
%%writefile daily_report.py
import json
from datetime import datetime
from pathlib import Path
from engine.data_feed import refresh_watchlist, load_stock
from engine.indicators import add_indicators
from engine.signals import scan_universe
from engine.tracker import open_trade, check_open_trades, print_scorecard, export_csv

def generate_report():
    today = datetime.now().strftime("%Y-%m-%d %H:%M")
    print(f"\n{'='*50}")
    print(f"  ARES DAILY REPORT — {today}")
    print(f"{'='*50}\n")

    with open("config/watchlist.json") as f:
        watchlist = json.load(f)

    print("[1] Refreshing data...")
    refresh_watchlist(watchlist["symbols"])

    print("\n[2] Checking open positions...")
    check_open_trades()

    print("\n[3] MARKET OVERVIEW")
    print("-" * 40)
    for sym in ["SPY", "QQQ", "IWM"]:
        df = load_stock(sym)
        df = add_indicators(df)
        latest = df.iloc[-1]
        prev = df.iloc[-2]
        change_pct = (float(latest['Close']) - float(prev['Close'])) / float(prev['Close']) * 100
        direction = "+" if change_pct > 0 else "-"
        print(f"  {sym}: ${float(latest['Close']):.2f} "
              f"{direction}{abs(change_pct):.2f}% | "
              f"RSI: {float(latest['rsi']):.0f}")

    print(f"\n[4] SIGNAL SCAN")
    print("-" * 40)
    signals = scan_universe()

    if signals:
        for s in signals:
            print(f"\n  * SIGNAL: {s['symbol']}")
            print(f"    Strategy: {s['strategy']}")
            print(f"    Price:    ${s['price']}")
            print(f"    RSI:      {s['rsi']}")
            print(f"    Volume:   {s['vol_ratio']}x average")
            print(f"    Strength: {s['strength']}")

            portfolio = 10000
            position_size = portfolio * 0.10
            shares = position_size / s['price']
            stop_loss = s['price'] - (s['price'] * s['stdev_20'] * 2)
            print(f"\n    --- WHAT TO DO ---")
            print(f"    Buy:       ${position_size:.0f} worth "
                  f"({shares:.1f} shares)")
            print(f"    Stop-loss: ${stop_loss:.2f}")
            print(f"    Exit when: RSI > 50")

            open_trade(s)
            print(f"    [Recorded as virtual trade]")
    else:
        print("  No signals today. Do nothing.")

    print_scorecard()
    export_csv()

    print(f"\n[6] WATCHLIST STATUS")
    print("-" * 40)
    print(f"  {'Symbol':<8}{'Price':<10}{'RSI':<8}{'Vol':<8}{'Note'}")
    print(f"  {'------':<8}{'-----':<10}{'---':<8}{'---':<8}{'----'}")

    for sym in watchlist["symbols"]:
        try:
            df = load_stock(sym)
            df = add_indicators(df)
            latest = df.iloc[-1]
            rsi = float(latest['rsi'])
            vol = float(latest['vol_ratio'])
            price = float(latest['Close'])

            note = ""
            if rsi < 35:
                note = "<- near oversold"
            elif rsi > 65:
                note = "<- overbought"
            if vol > 1.8:
                note += " HIGH VOL"

            print(f"  {sym:<8}${price:<9.2f}{rsi:<8.0f}{vol:<8.1f}{note}")
        except Exception as e:
            print(f"  {sym:<8} ERROR: {e}")

    print(f"\n{'='*50}")
    print("  RULES REMINDER:")
    print("  - Max 10% of portfolio per trade")
    print("  - Keep 30% cash at all times")
    print("  - Sell at stop-loss — no exceptions")
    print("  - Max 8 positions open")
    print("  - Stop trading if down 5% this week")
    print(f"{'='*50}\n")

if __name__ == "__main__":
    generate_report()

In [ ]:
# Cell 8: RUN DAILY REPORT
# ========================
# Run this cell every day!
# ========================

%run daily_report.py

In [ ]:
# Cell 9 (Optional): Download CSV to your device
from google.colab import files
import os

csv_path = 'logs/trades_report.csv'
if os.path.exists(csv_path):
    files.download(csv_path)
    print('CSV downloaded!')
else:
    print('No trades yet — run Cell 8 first and wait for signals.')

In [ ]:
# Cell 10 (Optional): View trade history
import json
import os

trades_path = 'logs/virtual_trades.json'
if os.path.exists(trades_path):
    with open(trades_path) as f:
        trades = json.load(f)
    print(f'Total trades: {len(trades)}')
    print(f'Open: {len([t for t in trades if t["status"] == "open"])}')
    print(f'Closed: {len([t for t in trades if t["status"] == "closed"])}')
    print('\nAll trades:')
    for t in trades:
        status = t['status'].upper()
        pnl = f"{t['pnl_pct']:+.1f}%" if t['pnl_pct'] else 'pending'
        print(f"  {t['symbol']} | {t['entry_date']} | ${t['entry_price']} | {status} | {pnl}")
else:
    print('No trades yet.')